In [ ]:
import sys
from pathlib import Path

project_root = Path.cwd().resolve().parent

src_path = project_root / 'src'
if str(src_path) not in sys.path:
    sys.path.append(str(src_path))
    
from config import *
print(f"Environment linked. Data dir: {DATA_DIR}")
print(f"Saving images: {SAVE_FIGURES}")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from plotting import *
setup_plotting_style()

In [ ]:
import joblib
import pandas as pd
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc

NB_ID = "40"

In [ ]:
# Load the Model

if not MACHINE_A_MODEL_FILE.exists():
    raise FileNotFoundError(f"Model not found at {MACHINE_A_MODEL_FILE}. Please run Notebook 30 first.")

model_pipeline = joblib.load(MACHINE_A_MODEL_FILE)
print(f"Loaded Model: {type(model_pipeline).__name__}")

if not MACHINE_A_TEST_SET_FILE.exists():
    raise FileNotFoundError(f"Test set not found at {MACHINE_A_TEST_SET_FILE}.")

# Load the Test Set
df_test = pd.read_csv(MACHINE_A_TEST_SET_FILE)

# We access the features from the pipeline scaler to ensure alignment
try:
    scaler_step = model_pipeline.named_steps['scaler']
    if hasattr(scaler_step, 'feature_names_in_'):
        features = list(scaler_step.feature_names_in_)
    else:
        features = MACHINE_A_FEATURES
except:
    features = MACHINE_A_FEATURES

X_test = df_test[features]
y_test = df_test['label']

print(f"Loaded Test Data: {len(df_test)} samples")
print(f"Features: {features}")

In [ ]:
# Extract the Logistic Regression step
log_reg = model_pipeline.named_steps['log_reg']

# Create Dataframe for plotting
coefs = pd.DataFrame({
    'Feature': features,
    'Weight': log_reg.coef_[0],
    'Abs_Weight': np.abs(log_reg.coef_[0])
}).sort_values(by='Abs_Weight', ascending=False)

# Plot
plt.figure(figsize=(10, 5))

sns.barplot(
    data=coefs, 
    x='Weight', 
    y='Feature', 
    hue='Feature',
    palette="viridis", 
    legend=False
)
plt.axvline(0, color='black', linestyle='--', linewidth=1)
plt.title('Machine A: Decision Logic (Feature Weights)')
plt.xlabel('Influence (Left=Clean, Right=Jamming)')

save_plot("01_feature_importance_physics_check", prefix=NB_ID)
plt.show()

# Fig 40.1 Feature Importance: The Physics of Detection

The Feature Importance analysis (Fig 40.1) provides critical insight into the model's decision-making logic, confirming that it has learned physical characteristics of RF interference rather than superficial correlations.

### The Dominance of "Spectral Flatness"
The most striking result is the overwhelming weight assigned to **Spectral Flatness** (the purple bar).
* **Observation:** This feature is the primary driver for classifying a signal as "Jamming."
* **Physical Interpretation:** This aligns perfectly with RF physics.
    * **Clean Signals (OFDM/WiFi):** Possess a distinct "City Skyline" structure in the frequency domain (peaks for sub-carriers, valleys for guard bands). They are **not** flat.
    * **Jamming (Barrage):** By definition, barrage jamming attempts to fill the entire frequency band with uniform noise. In the frequency domain, this appears as a "Flat Desert."
* **Conclusion:** The model correctly identified that **texture** (how the signal looks) is a more reliable indicator of malice than volume.

### The Unreliability of "Mean Power"
Crucially, **Mean Power** (the green bar) is only the 4th most important feature.
* **Why this is positive:** A naive model often over-relies on power, flagging any loud signal as jamming.
* **Our Model's Logic:** By ranking Power lower than Flatness and PAPR, the model acknowledges that **loud signals can be clean** (e.g., a legitimate transmitter nearby) and **quiet signals can be jammers** (e.g., a jammer far away).
* **Operational Benefit:** This makes the system robust against false alarms caused by legitimate high-power transmissions.

### The Role of PAPR
**PAPR (Peak-to-Average Power Ratio)** emerges as the secondary discriminator.
* **Physics:** Jamming signals often exhibit constant-envelope characteristics (low PAPR) or specific statistical distributions that differ from the bursty, high-variance nature of modulated communication signals.
* **Logic:** The model effectively uses a "Two-Key System":
    1.  **Check 1:** Is the spectrum unnaturally flat? (Spectral Flatness)
    2.  **Check 2:** Does the power distribution look "machine-consistent"? (PAPR)

**Verdict:** The decision logic is scientifically sound. The model has autonomously discovered that **Spectral Shape** is the true fingerprint of interference.

In [ ]:
# Generate Predictions
y_pred = model_pipeline.predict(X_test)
y_prob = model_pipeline.predict_proba(X_test)[:, 1]

# ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_prob)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(6, 6))
plt.plot(fpr, tpr, color='C1', lw=3, label=f'AUC = {roc_auc:.3f}')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.xlabel('False Alarm Rate')
plt.ylabel('Detection Rate')
plt.title('ROC Curve')
plt.legend(loc="lower right")

save_plot("02_roc_curve", prefix=NB_ID)
plt.show()


#  Fig 40.2 Performance Metrics: ROC & AUC Analysis

The Receiver Operating Characteristic (ROC) curve provides a comprehensive view of the trade-off between sensitivity and specificity.

### 1. Global Performance (AUC = 0.962)
The model achieves an Area Under the Curve (AUC) of **0.962**, classifying it as an "Excellent" discriminator according to standard statistical benchmarks.
* **Random Guess:** AUC = 0.5 (Diagonal Line).
* **Perfect Model:** AUC = 1.0.
* **Our Model:** 0.962. This indicates that given a random pair of (Jamming, Clean) samples, the model will correctly rank the Jamming sample higher than the Clean one 96.2% of the time.

### 2. Operational Sweet Spot (The "Vertical Rise")
The most critical feature of this curve is the steep vertical ascent at the start.
* **Observation:** The True Positive Rate (Detection) reaches $\approx$ 90% while the False Positive Rate (False Alarm) remains near 0.05.
* **Implication:** The system can detect the vast majority of threats (likely the high-power ones) with an extremely low rate of false alarms. This is ideal for an automated edge system, where false alarms cause "alert fatigue."

### 3. The "Cost" of Perfection
The curve flattens out at the top-right, indicating diminishing returns.
* To increase detection from 95% to 100% (catching the weak 0.1 power jammers), we would have to accept a significantly higher False Alarm Rate.
* This confirms our findings from Section 4.4: The last few percent of "missing" jammers are physically indistinguishable from clean thermal noise. Forcing the model to catch them would force it to hallucinate jammers in clean noise.

**Verdict:** The ROC profile confirms that the model is **highly robust** for its intended operating range, failing only in the regime where discrimination is physically ambiguous.

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False,
            xticklabels=['Pred Clean', 'Pred Jammed'],
            yticklabels=['True Clean', 'True Jammed'])
plt.title('Confusion Matrix (Test Set)')

save_plot('03_confusion_matrix', prefix=NB_ID)
plt.show()

print("\nDetailed Classification Report:")
print(classification_report(y_test, y_pred))

# Fig 40.3 Classification & Confusion Matrix Analysis

The Confusion Matrix (Fig 40.3) and Classification Report provide the final validation of the system's operational reliability.

### 1. High-Trust Alerts (Precision: 96%)
The most operationally significant metric is the **96% Precision** for the Jamming class.
* **Data:** Out of ~42,000 alerts generated by the system, only **1,530** were false alarms.
* **Implication:** The system exhibits a very low False Positive Rate. This is critical for automated response systems (e.g., frequency hopping), as false alarms would disrupt legitimate communications unnecessarily.

### 2. Analysis of Missed Detections (The "False Negatives")
The matrix shows **5,952** False Negatives (Jammers classified as Clean).
* **Context:** While this lowers the Recall to 87%, cross-referencing this with the Sensitivity Analysis (Fig 40.4) confirms that these misses are concentrated in the **0.1 Power** regime.
* **Verdict:** The system effectively filters out "nuisance" low-power interference, focusing detection on high-confidence, high-power threats.

### Final Verdict: Machine A
The **Logistic Regression** baseline has successfully demonstrated:
1.  **Physics-Aware Learning:** Prioritizing Spectral Flatness over Raw Power.
2.  **Operational Robustness:** High precision and low false alarm rates.
3.  **Defined Sensitivity:** A clear operational floor at 0.3 relative power.


In [ ]:
results = df_test.copy()
results['y_pred'] = y_pred
results['is_correct'] = (results['label'] == results['y_pred'])

# Filter for JAMMING samples only (we care about Recall here)
jamming_only = results[results['label'] == 1]

# Analyze Accuracy vs Jamming Power
power_acc = jamming_only.groupby('jamming_power')['is_correct'].mean().reset_index()

plt.figure(figsize=(8, 4))
plot = sns.barplot(
    data=power_acc, 
    x='jamming_power', 
    y='is_correct', 
    hue='jamming_power', 
    palette='magma', 
    legend=False
)
plt.axhline(0.9, color='red', linestyle='--', label='90% Target')

plt.title('Detection Sensitivity vs. Signal Strength')
plt.ylabel('Recall (Detection Rate)')
plt.xlabel('Jamming Power (Relative)')
plt.ylim(0, 1.15)
plt.legend()

# Label bars with exact values
for p in plot.patches:
    if p.get_height() > 0:
        plot.annotate(f'{p.get_height():.2f}', 
                      (p.get_x() + p.get_width() / 2., p.get_height()), 
                      ha='center', va='bottom', fontsize=9, xytext=(0, 2), 
                      textcoords='offset points')

save_plot('04_sensitivity_vs_power', prefix=NB_ID)
plt.show()

# Fig 40.4 Conclusion: The Physics of Detection Sensitivity

The detailed sensitivity analysis reveals a distinct **Operational Sensitivity Floor** for the statistical detection approach.

### 1. The "Sensitivity Cliff" at Power 0.1
We observe a sharp drop in recall for jamming signals with relative power $\le 0.1$.
* **Observation:** The model detects high-power interference ($>0.3$) with **>99% accuracy**, but performance degrades to near-random guessing at power level 0.1.
* **Physical Explanation:** This is **expected behavior** governed by the Signal-to-Noise Ratio (SNR). At power level 0.1, the interference energy is comparable to or lower than the thermal noise floor of the USRP X310 hardware.
* **Feature Impact:** Our primary features (Spectral Flatness, Kurtosis) rely on the signal disrupting the Gaussian distribution of thermal noise. When the signal is buried in the noise floor, these statistical moments remain unchanged, making the jammer mathematically invisible to this lightweight architecture.

### 2. Validation of Model Integrity
Paradoxically, this "failure" at low power serves as a strong validation of the model's integrity:
* If the model achieved high accuracy at Power 0.1 (where the signal is physically invisible), it would indicate **overfitting** or **data leakage** (e.g., learning metadata or filenames instead of signal physics).
* The fact that the model fails exactly where physics predicts it should confirms it is learning valid RF characteristics.

### 3. Operational Viability
From a system perspective, this sensitivity floor is acceptable:
* **Destructive Interference:** High-power jammers that disrupt communication are detected with near-perfect reliability.
* **Negligible Interference:** Low-power signals (0.1) that are missed by the model are likely too weak to significantly degrade the Bit Error Rate (BER) of a robust receiver.

**Verdict:** Machine A successfully balances computational efficiency (running in microseconds) with effective detection of all operationally significant jamming threats.

In [ ]:
# Analyze Accuracy vs Jammer Type
type_acc = jamming_only.groupby('jammer_type')['is_correct'].mean().reset_index()

plt.figure(figsize=(6, 4))
sns.barplot(
    data=type_acc, 
    x='jammer_type', 
    y='is_correct', 
    hue='jammer_type',
    palette='viridis', 
    legend=False
)
plt.title('Detection Robustness vs. Jammer Type')
plt.ylabel('Recall')
plt.ylim(0, 1.1)

save_plot('05_sensitivity_vs_type', prefix=NB_ID)
plt.show()

# 4.5 Generalization Analysis: Robustness Across Jammer Types

The final analysis (Fig 40.5) evaluates whether the model is biased toward specific jamming techniques.

### 1. Consistent Performance
The model demonstrates remarkable consistency across the two primary jamming classes:
* **Gaussian Noise (Barrage):** $\approx$ 89% Recall.
* **Sinusoidal (CW):** $\approx$ 85% Recall.

### 2. Validation of Feature Selection
This result validates the decision to use a composite feature set:
* **Gauss Detection:** Likely driven by **Spectral Flatness**, as Gaussian noise is the textbook definition of a flat spectrum.
* **Sin Detection:** Likely driven by **PAPR** and **Kurtosis**. A sinusoidal jammer is not spectrally flat (it appears as a peak), so a model relying solely on flatness would fail. The high recall here proves the model effectively utilizes secondary statistical moments to identify narrowband threats.